<a href="https://colab.research.google.com/github/priya29o08/Flask-personal-api/blob/main/car_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Car Price Prediction Model with Interactive Interface
# Upload your Cleaned_Car_data.csv file when prompted

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. LOAD DATA
print(" Upload your Cleaned_Car_data.csv file...")
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('Cleaned_Car_data.csv')
print(f"\n Data loaded: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(df.head())
print("\n" + "="*50)

# 2. DATA PREPROCESSING
print("\n🔧 Preprocessing data...")
df_model = df.copy()

# Encode categorical variables
le_company = LabelEncoder()
le_fuel = LabelEncoder()
le_name = LabelEncoder()

df_model['company_encoded'] = le_company.fit_transform(df_model['company'])
df_model['fuel_type_encoded'] = le_fuel.fit_transform(df_model['fuel_type'])
df_model['name_encoded'] = le_name.fit_transform(df_model['name'])

# Select features for modeling
X = df_model[['company_encoded', 'name_encoded', 'year', 'kms_driven', 'fuel_type_encoded']]
y = df_model['Price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f" Training set: {X_train.shape[0]} samples")
print(f" Test set: {X_test.shape[0]} samples\n")

# 3. TRAIN MODELS
print(" Training models...\n")

# Random Forest (typically performs better)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)

print(" MODEL PERFORMANCE")
print("="*50)
print(f"Random Forest - R² Score: {rf_r2:.3f}, MAE: ₹{rf_mae:,.0f}")
print("="*50)

# 4. VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, rf_pred, alpha=0.6, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price (₹)')
axes[0].set_ylabel('Predicted Price (₹)')
axes[0].set_title('Actual vs Predicted Prices')
axes[0].grid(alpha=0.3)

# Feature Importance
importances = rf_model.feature_importances_
features = ['Company', 'Model', 'Year', 'KMs Driven', 'Fuel Type']
axes[1].barh(features, importances, color='green')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance')

plt.tight_layout()
plt.show()

# 5. INTERACTIVE PREDICTION INTERFACE
print("\n" + "="*60)
print(" CAR PRICE PREDICTION INTERFACE")
print("="*60 + "\n")

# Create dropdowns with sorted unique values
company_dropdown = widgets.Dropdown(
    options=sorted(df['company'].unique()),
    description='Company:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Model dropdown - updates based on company selection
model_dropdown = widgets.Dropdown(
    options=sorted(df['name'].unique()),
    description='Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

year_dropdown = widgets.Dropdown(
    options=sorted(df['year'].unique(), reverse=True),
    description='Year:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

fuel_dropdown = widgets.Dropdown(
    options=sorted(df['fuel_type'].unique()),
    description='Fuel Type:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

kms_input = widgets.IntText(
    value=50000,
    description='KMs Driven:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Predict button
predict_button = widgets.Button(
    description=' PREDICT PRICE',
    button_style='success',
    layout=widgets.Layout(width='400px', height='50px'),
    style={'font_weight': 'bold'}
)

# Output area
output = widgets.Output()

# Prediction function
def predict_price(b):
    with output:
        output.clear_output()

        try:
            # Get values
            company = company_dropdown.value
            model_name = model_dropdown.value
            year = year_dropdown.value
            kms = kms_input.value
            fuel = fuel_dropdown.value

            # Encode inputs
            company_enc = le_company.transform([company])[0]
            model_enc = le_name.transform([model_name])[0]
            fuel_enc = le_fuel.transform([fuel])[0]

            # Make prediction
            input_data = [[company_enc, model_enc, year, kms, fuel_enc]]
            predicted_price = rf_model.predict(input_data)[0]

            # Display result with styling
            display(HTML(f"""
                <div style='background-color: #e8f5e9; padding: 20px; border-radius: 10px;
                            border: 3px solid #4caf50; margin-top: 20px;'>
                    <h2 style='color: #2e7d32; margin: 0;'> Estimated Price</h2>
                    <h1 style='color: #1b5e20; margin: 10px 0; font-size: 48px;'>
                        ₹ {predicted_price:,.0f}
                    </h1>
                    <p style='color: #558b2f; margin: 0; font-size: 14px;'>
                        {company} {model_name} | {year} | {kms:,} kms | {fuel}
                    </p>
                </div>
            """))

        except Exception as e:
            display(HTML(f"""
                <div style='background-color: #ffebee; padding: 20px; border-radius: 10px;
                            border: 3px solid #f44336;'>
                    <h3 style='color: #c62828; margin: 0;'> Error</h3>
                    <p style='color: #d32f2f;'>{str(e)}</p>
                </div>
            """))

# Update model dropdown based on company selection
def update_models(change):
    company = change['new']
    available_models = sorted(df[df['company'] == company]['name'].unique())
    model_dropdown.options = available_models

company_dropdown.observe(update_models, names='value')

# Connect button to prediction function
predict_button.on_click(predict_price)

# Display interface
display(HTML("<h3>Select Car Details:</h3>"))
display(company_dropdown)
display(model_dropdown)
display(year_dropdown)
display(fuel_dropdown)
display(kms_input)
display(predict_button)
display(output)

print("\n Select options and click PREDICT PRICE button.")

 Upload your Cleaned_Car_data.csv file...
